# Per-string CometKiwi scores (Colab GPU)

Scores every distinct (English source, translation) pair from the two upstream benchmark repositories with `Unbabel/wmt22-cometkiwi-da`, the same model and library versions used upstream. Nothing is translated.

## Before you start (once)
1. **Hugging Face:** sign in at huggingface.co, open https://huggingface.co/Unbabel/wmt22-cometkiwi-da and accept the licence (gated model). Then create a **read** token (Settings → Access Tokens).
2. **Colab secret:** in this notebook, click the 🔑 key icon on the left → *Add new secret* → name `HF_TOKEN`, value = your token → enable *Notebook access*. Never paste the token into a cell.
3. **Google Drive:** a folder named `hybrid-router-study` directly in *My Drive* (no `/` in the name), containing:
   - `score_pairs.py`
   - a `data` folder with `pairs_main.jsonl.gz`, `pairs_rest.jsonl.gz`, `parity_check.jsonl.gz`, `manifest.json`

   (Step 3 also finds `score_pairs.py` one or two folders deeper, and accepts the four files next to it instead of in `data/`.)
4. **GPU:** *Runtime → Change runtime type → T4 GPU*.

Results are saved to `hybrid-router-study/scores/` on Drive after every chunk of 20,000 pairs. If Colab disconnects, reconnect and run all cells again: finished chunks are skipped.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the scorer at the upstream versions (unbabel-comet 2.2.7, transformers 4.57.6, numpy 1.26.4),
#    then remove Colab's preinstalled TensorFlow/Keras and JAX/Flax. They need newer numpy/protobuf than
#    CometKiwi and crash when transformers tries to import them. Nothing in this notebook uses them.
#    The pip "dependency conflicts" warnings are about other preinstalled packages; ignore them.
#    AFTERWARDS: Runtime -> Restart session, then continue from step 3 (not this cell).
!pip install -q "unbabel-comet==2.2.7" "transformers==4.57.6" "setuptools<81"
!pip uninstall -y -q tensorflow tensorboard tf-keras keras tensorflow-text tensorflow-hub tensorflow-decision-forests ydf jax jaxlib flax optax chex orbax-checkpoint

In [ ]:
# 3. Mount Drive, find the folder, check the input files against the manifest, import the scoring module
import glob, hashlib, json, os, sys
os.environ["USE_TF"] = "0"      # make transformers ignore TensorFlow
os.environ["USE_TORCH"] = "1"
os.environ["USE_FLAX"] = "0"    # ... and JAX/Flax
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive"
hits = sorted(glob.glob(f"{ROOT}/*/score_pairs.py") + glob.glob(f"{ROOT}/*/*/score_pairs.py"))
assert hits, "score_pairs.py not found in My Drive (top two folder levels). Check the upload and folder name."
assert len(hits) == 1, f"Found more than one score_pairs.py, keep one: {hits}"
BASE = os.path.dirname(hits[0])                       # folder holding score_pairs.py
DATA = os.path.join(BASE, "data") if os.path.exists(os.path.join(BASE, "data", "manifest.json")) else BASE
OUT = os.path.join(BASE, "scores")
print("code folder:", BASE, "\ndata folder:", DATA, "\nresults to: ", OUT)

sys.path.insert(0, BASE)
import score_pairs as sp

manifest = json.load(open(os.path.join(DATA, "manifest.json")))
for name, info in manifest.items():
    digest = hashlib.sha256(open(os.path.join(DATA, name), "rb").read()).hexdigest()
    assert digest == info["sha256"], f"{name} does not match the manifest: re-upload it"
    print(f"OK  {name}: {info['items']:,} items")

In [ ]:
# 4. Log in to Hugging Face with the HF_TOKEN secret
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
# 5. Load CometKiwi and time a sample to estimate the run
import torch
predict = sp.load_cometkiwi(batch_size=64, gpus=1 if torch.cuda.is_available() else 0)
rate = sp.time_sample(predict, os.path.join(DATA, "pairs_main.jsonl.gz"), n=2000)
total = manifest["pairs_main.jsonl.gz"]["items"] + manifest["pairs_rest.jsonl.gz"]["items"]
print(f"{rate:.0f} pairs/s -> main set about {manifest['pairs_main.jsonl.gz']['items'] / rate / 60:.0f} min, "
      f"everything about {total / rate / 60:.0f} min")

In [ ]:
# 6. Parity check: two whole catalogs must reproduce the published upstream means (within 0.002).
#    Do not continue if this fails.
report = sp.parity_check(predict, os.path.join(DATA, "parity_check.jsonl.gz"))
for r in report:
    print(r)
json.dump(report, open(os.path.join(BASE, "parity_report.json"), "w"), indent=2)
assert all(r["ok"] for r in report), "Parity check failed: scores differ from the published upstream means"
print("Parity check passed")

In [ ]:
# 7. Score the main comparison set (NLLB + GPT-4o-mini + Llama-8B)
sp.score_file(predict, os.path.join(DATA, "pairs_main.jsonl.gz"), OUT)

In [ ]:
# 8. Score all remaining pairs (other NMT systems and LLMs)
sp.score_file(predict, os.path.join(DATA, "pairs_rest.jsonl.gz"), OUT)

## After it finishes
Download the folder `My Drive/hybrid-router-study/scores/` (and `parity_report.json` next to it) and put it in the repository as `scoring/scores/`. Then run on your laptop:

```bash
.venv/bin/python scripts/join_scores.py
```

It maps every score back to all 1,779,672 catalog rows and checks that each of the 174 catalogs reproduces the published upstream CometKiwi mean.